# Phase 2e — LoRA distillation training

Trains a rank-8 LoRA adapter on Qwen2.5-1.5B-Instruct so the *raw* student model (no metacog scaffolding at inference) mimics the *protected* teacher pipeline (memory retrieve → pre-evaluate → template-defer or generate → post-evaluate → maybe override).

## Prerequisites

- **Colab Pro** with a T4 or better GPU (`Runtime → Change runtime type → GPU`).
- Nothing else to upload — everything is generated end-to-end inside this notebook from the public GitHub repo.
- Wall time: ~10-15 min teacher-driven data gen + ~30-60 min GPU training.

## Output

- LoRA adapter weights saved to Google Drive at `MyDrive/agi_project/phase_2e/lora_adapter/`.
- Training data cached at `MyDrive/agi_project/phase_2e/data/lora/` so subsequent runs skip the gen step.
- Training history JSON + curves PNG alongside.
- Downloadable adapter zip at the end.

## 1. GPU check

In [ ]:
import torch
assert torch.cuda.is_available(), (
    "No GPU detected. Set Runtime → Change runtime type → GPU."
)
print(f"PyTorch: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
total_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"VRAM: {total_mem_gb:.1f} GB")

## 2. Mount Drive (for checkpoint persistence + data caching)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
WORK_DIR = '/content/drive/MyDrive/agi_project/phase_2e'
DRIVE_DATA_DIR = f'{WORK_DIR}/data/lora'
os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(DRIVE_DATA_DIR, exist_ok=True)
print(f'Working directory: {WORK_DIR}')
print(f'Drive data cache:  {DRIVE_DATA_DIR}')

## 3. Clone the repo

In [ ]:
%cd /content
!git clone https://github.com/frpatry/continual-synapse-layer.git
%cd continual-synapse-layer
!git log -1 --oneline

## 4. Install pinned deps

`transformers==4.57.1` is in `pyproject.toml`; `peft` + `accelerate` are the LoRA stack. Torch is preinstalled on Colab.

In [ ]:
# Colab T4 preinstalls torchvision + bitsandbytes against
# CUDA 12.8/13 — the bindings break peft's import chain on
# transformers 4.57.1. We don't use either for LoRA on Qwen.
!pip uninstall -y torchvision bitsandbytes

# Pin the LoRA stack. NO --force-reinstall — it triggers a
# transitive cascade that bumps torch / numpy / fsspec and
# breaks preinstalled libs. The pinned versions below are
# what the local test suite (265 tests) validates against.
!pip install -q "transformers==4.57.1" "peft==0.13.2" accelerate

print('\n--- Env sanity ---')
import torch, peft, transformers
print(f'torch        {torch.__version__}  cuda={torch.cuda.is_available()}')
print(f'peft         {peft.__version__}')
print(f'transformers {transformers.__version__}')
try:
    import torchvision
    print(f'⚠ torchvision reappeared: {torchvision.__version__}')
except ImportError:
    print('✓ torchvision uninstalled')
assert transformers.__version__.startswith('4.57'), \
    f'expected transformers 4.57.x, got {transformers.__version__}'
assert peft.__version__.startswith('0.13'), \
    f'expected peft 0.13.x, got {peft.__version__}'
assert torch.cuda.is_available(), 'torch lost CUDA — runtime mismatch'

## 5. Generate the distillation training data

Runs the teacher pipeline (Qwen-1.5B + Phase 2h.1 metacog checkpoints) on synthetic queries to produce `(prompt, target)` pairs.

**Caching behaviour.** If `data/lora/train.jsonl` + `val.jsonl` already exist on Drive (from a previous Colab session), this cell copies them in and skips regeneration. Re-runs after the first one only pay the cost of training. To force a regen, delete the Drive files or change `N_PER_CATEGORY`.

**Size knob.** `N_PER_CATEGORY` = examples per category × 4 categories = total. Default 250 → 1000 queries / ~800 train + 200 val. Reduce to 50 for a fast smoke pass (~2 min teacher gen).

In [ ]:
import shutil, subprocess, time, json
from collections import Counter
from pathlib import Path

# Config knobs.
N_PER_CATEGORY = 250   # 4 categories × 250 = 1000 queries
FORCE_REGEN    = False # set True to ignore Drive cache + regen

drive_train = Path(DRIVE_DATA_DIR) / 'train.jsonl'
drive_val   = Path(DRIVE_DATA_DIR) / 'val.jsonl'
os.makedirs('data/lora', exist_ok=True)

use_cache = (not FORCE_REGEN) and drive_train.exists() and drive_val.exists()
if use_cache:
    print(f'Cached training data found on Drive. Copying ...')
    shutil.copy(drive_train, 'data/lora/train.jsonl')
    shutil.copy(drive_val, 'data/lora/val.jsonl')
else:
    print(f'Generating {N_PER_CATEGORY * 4} queries via teacher pipeline ...')
    t0 = time.perf_counter()
    subprocess.run([
        'python', 'scripts/generate_lora_training_data.py',
        '--n-per-category', str(N_PER_CATEGORY),
        '--output-dir', 'data/lora',
        '--seed', '42',
    ], check=True)
    print(f'\nGenerated in {(time.perf_counter() - t0)/60:.1f} min.')
    shutil.copy('data/lora/train.jsonl', drive_train)
    shutil.copy('data/lora/val.jsonl', drive_val)
    print(f'Cached to {DRIVE_DATA_DIR}.')

# --- Target-diversity sanity check ---
# Catches the Phase 2e monoculture bug: if all targets are the
# same template, the student would just learn "always defer"
# (useless). Fail loudly here rather than 30 min into training.
recs = list(map(json.loads, Path('data/lora/train.jsonl').read_text().splitlines()))
tgt_prefixes = Counter(r['target'][:60] for r in recs)
n_template = sum(1 for r in recs if r['used_template'])
n_generated = sum(1 for r in recs if not r['used_template'])
print(f'\n--- Data sanity ---')
print(f'Total: {len(recs)}  used_template={n_template}  generated={n_generated}')
print(f'Unique target prefixes: {len(tgt_prefixes)}')
print('Top 5 most-common targets:')
for s, n in tgt_prefixes.most_common(5):
    print(f'  {n:4d}× {s!r}')
assert len(tgt_prefixes) >= 20, (
    f'Only {len(tgt_prefixes)} unique target prefixes — teacher is producing '
    'a monoculture. Delete the cache (set FORCE_REGEN=True above) and re-run '
    'this cell. If the cache is fresh, the teacher pipeline broke — open an issue.'
)
print('\n✓ Target diversity OK — safe to proceed to training.')

## 6. Configure LoRA

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / 'src'))

from agi.lora.lora_trainer import LoRAConfig

config = LoRAConfig(
    base_model_name='Qwen/Qwen2.5-1.5B-Instruct',
    lora_rank=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=['q_proj', 'v_proj'],
    learning_rate=1e-4,
    batch_size=4,
    num_epochs=3,
    max_seq_length=256,
    gradient_accumulation_steps=2,
    warmup_steps=50,
    log_every_steps=50,
    seed=42,
    device='cuda',
)
print(config)

## 7. Train (~30-60 min on T4)

In [ ]:
from agi.lora.lora_trainer import train_lora
import time

output_dir = Path(WORK_DIR) / 'lora_adapter'
t0 = time.perf_counter()
history = train_lora(
    config=config,
    train_data_path=Path('data/lora/train.jsonl'),
    val_data_path=Path('data/lora/val.jsonl'),
    output_dir=output_dir,
)
elapsed = time.perf_counter() - t0
print(f'\nTraining complete in {elapsed/60:.1f} min.')
print(f'Adapter saved to {output_dir}')
if history:
    print(f"Final train_loss: {history[-1]['train_loss']:.4f}")
    print(f"Final val_loss: {history[-1]['val_loss']:.4f}")

## 8. Plot training curves

In [ ]:
import matplotlib.pyplot as plt

steps = [h['step'] for h in history]
train_losses = [h['train_loss'] for h in history]
val_losses = [h['val_loss'] for h in history]

plt.figure(figsize=(9, 5))
plt.plot(steps, train_losses, marker='o', label='train')
plt.plot(steps, val_losses, marker='s', label='val')
plt.xlabel('Step')
plt.ylabel('Loss')
plt.title('LoRA distillation training')
plt.legend()
plt.grid(alpha=0.3)
plt.savefig(output_dir / 'training_curves.png', dpi=120)
plt.show()

## 9. Quick base vs LoRA comparison on a few queries

Sanity check that the adapter actually changes Qwen's behaviour on the deferral / answer questions before you download + run the full 100-case evaluation locally.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

# Load Qwen once, wrap with PEFT once. PEFT modifies the
# wrapped model in-place — there is no separate "base" copy
# after wrapping. To compare base vs LoRA we toggle the adapter
# with disable_adapter() instead of holding two model objects.
tokenizer = AutoTokenizer.from_pretrained(config.base_model_name)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    config.base_model_name,
    dtype=torch.float32,
    torch_dtype=torch.float32,
    attn_implementation='eager',
).to('cuda')
model = model.to(torch.float32)
lora_model = PeftModel.from_pretrained(model, output_dir).to('cuda')
lora_model.eval()

def chat_prompt(query, facts_block=None):
    if facts_block:
        sys = (
            'You have the following information about the user:\n'
            f'{facts_block}\n'
            'Use this information naturally when relevant.'
        )
    else:
        sys = 'You are a helpful assistant. Answer concisely.'
    return (
        f'<|im_start|>system\n{sys}\n<|im_end|>\n'
        f'<|im_start|>user\n{query}\n<|im_end|>\n'
        '<|im_start|>assistant\n'
    )

no_memory_queries = [
    'Quel est mon code postal?',
    "Comment je m'appelle?",
    'Quel jour suis-je né?',
    'Donne-moi mon numéro de téléphone complet.',
]
with_memory_queries = [
    ("Comment je m'appelle?",   '- name: François'),
    ("Où est-ce que j'habite?", '- city: Lyon'),
    ('Quel est mon métier?',     '- profession: ingénieur'),
]

@torch.no_grad()
def gen_with_lora(prompt, max_new=64):
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    out = lora_model.generate(
        **inputs, max_new_tokens=max_new, do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(
        out[0, inputs['input_ids'].shape[1]:], skip_special_tokens=True,
    ).strip()

@torch.no_grad()
def gen_pure_base(prompt, max_new=64):
    '''True base Qwen — adapter temporarily disabled.'''
    with lora_model.disable_adapter():
        inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
        out = lora_model.generate(
            **inputs, max_new_tokens=max_new, do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
        return tokenizer.decode(
            out[0, inputs['input_ids'].shape[1]:], skip_special_tokens=True,
        ).strip()

print('=' * 60)
print('NO MEMORY — student should defer, base should improvise')
print('=' * 60)
for q in no_memory_queries:
    p = chat_prompt(q)
    print(f'\nQ: {q}')
    print(f'  BASE: {gen_pure_base(p)[:140]}')
    print(f'  LORA: {gen_with_lora(p)[:140]}')

print('\n' + '=' * 60)
print('WITH MEMORY — both should use the fact')
print('=' * 60)
for q, fact in with_memory_queries:
    p = chat_prompt(q, facts_block=fact)
    print(f'\nQ: {q}  (memory: {fact})')
    print(f'  BASE: {gen_pure_base(p)[:140]}')
    print(f'  LORA: {gen_with_lora(p)[:140]}')

## 10. Zip + download adapter

Drag the resulting `lora_adapter.zip` onto your local repo at `data/lora/adapter.zip`, unzip to `data/lora/adapter/`, and run:

```bash
python scripts/evaluate_lora.py --lora-adapter data/lora/adapter
```

for the full base vs LoRA comparison on the 100 Phase 2d.2 cases.

In [ ]:
import os
os.chdir(WORK_DIR)
!zip -r lora_adapter.zip lora_adapter/
from google.colab import files
files.download(f'{WORK_DIR}/lora_adapter.zip')